<a href="https://colab.research.google.com/github/fatoufall737/atelier-tensorflow-iot/blob/main/atelier_tensorflow_iot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

Partie 1 – Génération du dataset
1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes :
a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un
écart-type de 4 °C.
b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %.
c) occupants : valeurs entières choisies entre 1 et 49 inclus.
2) Déterminer la variable consommation avec la formule suivante :
 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50
 chaque degré supplémentaire augmente la consommation de 5 unités
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation
 chaque personne présente dans la pièce augmente la consommation de 4 unités
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus
ou d'autres facteurs non mesurés.
3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au
format (float32) optimisé pour les calculs
4)  Créer la cible (target) y qui contiendra la variable consommation, au format float32

In [4]:
#Question 1
#a : génère 1000 valeurs de température (loi normale, moyenne 25°C, écart-type 4°C).
np.random.seed(42)  # pour la reproductibilité
n = 1000
temperature = np.random.normal(loc=25, scale=4, size=n)
temperature[:5]

array([26.98685661, 24.4469428 , 27.59075415, 31.09211943, 24.0633865 ])

In [5]:
#b : génère 1000 valeurs d'humidité (réparties uniformément entre 30% et 80%).
humidite = np.random.uniform(low=30, high=80, size=n)
humidite[:5]

array([38.37412911, 35.22839202, 61.82151248, 65.32378632, 31.57930724])

In [6]:
#c : génère 1000 valeurs d'occupants (entiers entre 1 et 49 inclus).
occupants = np.random.randint(low=1, high=50, size=n)
occupants[:5]

array([ 8, 43, 20, 33, 26])

In [7]:
#question 2 : détermine la variable consommation avec la formule donnée.
bruit = np.random.normal(loc=0, scale=10, size=n)

consommation = 50 + (5 * temperature) + (1.5 * humidite) + (4 * occupants) + bruit
consommation[:5]

array([281.62608854, 406.67868707, 348.62841819, 433.63621523,
       318.9414931 ])

In [8]:
#Question 3 : rassemble temperature, humidite, occupants dans la matrice X (1000x3), en float32.
X = np.column_stack([temperature, humidite, occupants]).astype(np.float32)
print(X.shape)
X[:5]

(1000, 3)


array([[26.986856, 38.37413 ,  8.      ],
       [24.446943, 35.228394, 43.      ],
       [27.590754, 61.821514, 20.      ],
       [31.09212 , 65.323784, 33.      ],
       [24.063387, 31.579308, 26.      ]], dtype=float32)

In [9]:
#Question 4 : crée la cible y (la consommation), en float32.
y = consommation.astype(np.float32)
print(y.shape)
y[:5]

(1000,)


array([281.6261 , 406.67868, 348.62842, 433.63623, 318.9415 ],
      dtype=float32)

Partie 2 – Découpage Train/Test
Diviser le dataset précédent (X et y) en deux ensembles distincts : un pour l'entraînement (train)
et un pour le test (test). Avec les conditions suivantes : 20% des données serviront au test ; garantir
la reproductibilité du découpage.

In [10]:
#Partie 2 : Découpage Train/Test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

(800, 3) (200, 3)


Partie 3 – Création du modèle
1) Construire un réseau de neurones qui utilise l’architecture séquentielle suivante :
a) couche 1 : 16 neurones avec relu comme fonction d’activation
b) couche 2 : 8 neurones avec relu
c) couche 3 : 1 seul neurone.
2) Afficher un résumé textuel de l'architecture du réseau de neurones.

In [11]:
#question 1 : construis le réseau de neurones avec l'architecture séquentielle demandée.
modele = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation="relu", input_shape=(3,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1)
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
#Question 2 (dernière de la Partie 3) : affiche un résumé textuel de l'architecture du réseau.

modele.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)